# 01 - BM25 retrieval

Pure sparse keyword retrieval using `rank_bm25`. Strong baseline for queries
that contain rare or exact-match terms (ticker symbols, defined accounting
terms, dollar figures). No embeddings, no GPU, no network.

Pipeline: load PDF -> chunk -> `BM25Retriever` -> ask the LLM.

In [1]:
import os
import sys
import warnings

sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv

from rag.data_ingestion import chunk_documents, load_documents
from rag.llm import LLM
from rag.metrics import embedding_faithfulness
from rag.pipeline import RAGPipeline
from rag.retrievers import BM25Retriever

warnings.filterwarnings('ignore')
load_dotenv(os.path.abspath('../.env'))

FILE_PATH = '../data/google_10K.pdf'
QUERY = 'List total revenues for every fiscal year reported in the consolidated statements of income, and show the year-over-year growth rate.'

docs = load_documents(FILE_PATH)
chunks = chunk_documents(docs, chunk_size=2000, chunk_overlap=200)
print(f'Loaded {len(docs)} pages -> {len(chunks)} chunks')

Loaded 107 pages -> 230 chunks


In [2]:
retriever = BM25Retriever()
retriever.add_documents(chunks)

for i, hit in enumerate(retriever.retrieve(QUERY, k=3), start=1):
    snippet = hit.document.page_content[:200].replace('\n', ' ')
    print(f'{i}. (BM25={hit.score:.3f}) {snippet}...')

1. (BM25=28.963) 2024 2025 $ Change % Change Consolidated revenues $ 350,018  $ 402,836  $ 52,818  15 % Cost of revenues $ 146,306  $ 162,535  $ 16,229  11 % Operating expenses $ 91,322  $ 111,262  $ 19,940  22 % Oper...
2. (BM25=27.625) Table of Contents Alphabet Inc. Provision for Income Taxes The following table presents provision for income taxes (in millions, except effective tax rate):   Year Ended December 31,   2024 2025 Incom...
3. (BM25=27.210) Table of Contents Alphabet Inc. • employee compensation expenses for employees in finance, human resources, information technology, legal, and other administrative support functions; • expenses relati...


In [3]:
llm = LLM(api_key=os.environ['GROQ_API_KEY']).get_llm(
    provider='groq',
    model_name='llama-3.3-70b-versatile',
    temperature=0.0,
)
pipeline = RAGPipeline(retriever=retriever, llm=llm, top_k=8)
response = pipeline.answer(QUERY)
print(response.answer)

The total revenues for the fiscal years reported are: 
$350,018 (2024) and $402,836 (2025). 
The year-over-year growth rate is 15% (calculated as $52,818 / $350,018) (Doc 1 | page=38).


## Inline evaluation

`embedding_faithfulness` is the maximum cosine between the LLM's answer and
any of the retrieved chunks. Higher means the answer is grounded in what
the retriever surfaced -- a quick LLM-free proxy for hallucination.

In [4]:
context_strings = [doc.page_content for doc in response.contexts]
faith = embedding_faithfulness(response.answer, context_strings)
print(f'embedding_faithfulness = {faith:.3f}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


embedding_faithfulness = 0.553
